# Deep Learning Transformer Models for WAF Attack Classification

## 1. Introduction

This notebook implements the **training and comparative evaluation** of multiple transformer-based deep learning classifiers for a **Web Application Firewall (WAF) attack detection system**.

### System Architecture

The classification models developed here operate within a multi-stage HTTP request analysis pipeline:

```
Incoming HTTP Request
  → Nginx (Reverse Proxy)
  → ModSecurity WAF
  → OWASP CRS Rule Engine
      ├── No Alert → Allow Request
      └── Suspicious Alert → Alert Event Created
          → Redis Queue / Message Broker
          → Worker Node
              → Request Preprocessing
              → Tokenization & Encoding
              → Deep Learning Injection Classifier (THIS NOTEBOOK)
              → Attack Type Prediction
              → Confidence Calibration (Temperature Scaling)
              → Calibrated Confidence Score
                  → Triage Policy Engine
                      ├── LOW Confidence (<50%)   → Passive Monitoring + Light Rate Limit
                      ├── MEDIUM Confidence (50–80%) → Aggressive Throttling + CAPTCHA
                      └── HIGH Confidence (>80%)  → Immediate Blocking + Firewall Update
```

### Objective

When the OWASP CRS Rule Engine flags a suspicious request, it is placed into a **Redis queue**. A **Worker Node** dequeues the alert, preprocesses the HTTP payload, and passes it through a **transformer-based NLP classifier** to predict the specific attack type.

This notebook trains and compares **three transformer architectures**:

| # | Model | HuggingFace Identifier |
|---|-------|------------------------|
| 1 | BERT | `bert-base-uncased` |
| 2 | DistilBERT | `distilbert-base-uncased` |
| 3 | MiniLM-L6 | `nreimers/MiniLM-L6-H384-uncased` |

Each model is **fully fine-tuned** (all layers trainable) using **mixed precision (FP16) training**, **gradient accumulation**, **class-weighted loss**, and **post-training temperature scaling** — optimized for a **RTX 3060 Laptop (6 GB VRAM)** GPU with per-model batch configurations.

### Hardware Context

| Component | Spec |
|-----------|------|
| GPU | RTX 3060 Laptop, ~140W TGP, **6GB GDDR6 VRAM** |
| CPU | Ryzen 7 6800H, 8C/16T, up to 4.7GHz |
| RAM | 16GB DDR5-4800 dual-channel |
| CUDA | Ampere architecture, 3rd-gen Tensor Cores (FP16 native) |

### Dataset

The dataset consists of HTTP request payloads pre-split into training, validation, and test sets stored as Parquet files (SRBH_clean_v3.1.0). Labels represent injection attack categories detected by the WAF pipeline.

---
## 2. Import Libraries

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers pandas numpy scikit-learn matplotlib seaborn pyarrow

In [ ]:
import os
import re
import json
import warnings
import random
import subprocess
import copy
from collections import OrderedDict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    log_loss,
)
from scipy.optimize import minimize_scalar

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 3. Load Dataset

The dataset is pre-split into **train**, **validation**, and **test** Parquet files. No further splitting is required.

In [ ]:
DATA_DIR = "v3_907k_cleaned_parquet"

df_train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
df_val   = pd.read_parquet(os.path.join(DATA_DIR, "validation.parquet"))
df_test  = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

print(f"Train samples      : {len(df_train):,}")
print(f"Validation samples : {len(df_val):,}")
print(f"Test samples       : {len(df_test):,}")
print(f"\nColumns: {list(df_train.columns)}")
print(f"\n--- Train Sample Rows ---")
df_train.head(5)

### Construct Combined Payload Text & Identify Label Column

The dataset stores HTTP request components in separate columns. We construct a single **combined payload text** by concatenating the method, path, body, and user agent — matching the format used by the Worker Node in the WAF pipeline.

The label column is `final_label`.

In [ ]:
# --- Construct combined payload text ---
TEXT_COL = "request_text"

def build_combined_payload(df):
    """Construct combined HTTP request text from component columns."""
    method  = df["request_http_method"].fillna("").astype(str)
    path    = df["request_http_request"].fillna("").astype(str)
    body    = df["request_body"].fillna("").astype(str)
    ua      = df["request_user_agent"].fillna("").astype(str)
    combined = (
        "METHOD: " + method +
        "\nPATH: " + path +
        "\nBODY: " + body +
        "\nUA: " + ua
    )
    return combined

df_train[TEXT_COL] = build_combined_payload(df_train)
df_val[TEXT_COL]   = build_combined_payload(df_val)
df_test[TEXT_COL]  = build_combined_payload(df_test)

# --- Derive correct 4-class label from CAPEC flag columns ---
# Per the implementation plan:
#   CAPEC-66  → "SQL Injection"
#   CAPEC-242 → "Code Injection"
#   Any other CAPEC attack flag set → "Other Attacks"
#   No attack flags → "Normal"

CAPEC_SQL       = "66 - SQL Injection"
CAPEC_CODE_INJ  = "242 - Code Injection"
CAPEC_NORMAL    = "000 - Normal"

# All CAPEC columns that represent "other" attack types
OTHER_ATTACK_COLS = [
    c for c in df_train.columns
    if c not in [CAPEC_SQL, CAPEC_CODE_INJ, CAPEC_NORMAL]
    and c[0].isdigit()  # CAPEC columns start with a digit
]

print(f"SQL column      : {CAPEC_SQL}")
print(f"Code Inj column : {CAPEC_CODE_INJ}")
print(f"Other attack cols ({len(OTHER_ATTACK_COLS)}):")
for c in OTHER_ATTACK_COLS:
    print(f"  - {c}")

LABEL_COL = "label_4class"

def assign_4class_label(row):
    """Map CAPEC flags to 4-class label scheme."""
    if row[CAPEC_SQL] == 1:
        return "SQL Injection"
    elif row[CAPEC_CODE_INJ] == 1:
        return "Code Injection"
    else:
        for col in OTHER_ATTACK_COLS:
            if row[col] == 1:
                return "Other Attacks"
        return "Normal"

for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    df[LABEL_COL] = df.apply(assign_4class_label, axis=1)

print(f"\n--- 4-Class Label Distribution ---")
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    print(f"\n{name}:")
    dist = df[LABEL_COL].value_counts().sort_index()
    for lbl, cnt in dist.items():
        print(f"  {lbl:25s} : {cnt:>8,}  ({cnt/len(df)*100:.1f}%)")

print(f"\nLabel column : '{LABEL_COL}'")
print(f"Unique labels: {sorted(df_train[LABEL_COL].unique())}")
print(f"\n--- Sample combined payload ---")
print(df_train[TEXT_COL].iloc[0][:300])

---
## 4. Dataset Inspection

Verify sample counts, class balance, and visualize the label distribution across all splits.

In [ ]:
# --- Sample Counts ---
split_info = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Samples": [len(df_train), len(df_val), len(df_test)],
})
print(split_info.to_string(index=False))

# --- Class Distribution Per Split ---
print("\n=== Class Distribution ===")
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    dist = df[LABEL_COL].value_counts().sort_index()
    print(f"\n{name}:")
    for lbl, cnt in dist.items():
        print(f"  {lbl:25s} : {cnt:>8,}  ({cnt/len(df)*100:.1f}%)")

In [ ]:
# --- Visualization: Label Distribution ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, df) in zip(axes, [("Train", df_train), ("Validation", df_val), ("Test", df_test)]):
    counts = df[LABEL_COL].value_counts().sort_index()
    colors = sns.color_palette("Set2", n_colors=len(counts))
    bars = ax.bar(range(len(counts)), counts.values, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=30, ha="right", fontsize=9)
    ax.set_title(f"{name} Set — Label Distribution", fontsize=12, fontweight="bold")
    ax.set_ylabel("Count")
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                f"{val:,}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: label_distribution.png")

---
## 5. Data Preprocessing

Cleaning steps applied to HTTP request payload text:
1. Remove null / empty rows.
2. Strip leading/trailing whitespace.
3. Collapse excessive whitespace.
4. Encode categorical labels as integers.

In [ ]:
def clean_text(text: str) -> str:
    """Clean HTTP request payload text."""
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)  # collapse whitespace
    return text


# Apply cleaning to all splits
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    before = len(df)
    df.dropna(subset=[TEXT_COL, LABEL_COL], inplace=True)
    df[TEXT_COL] = df[TEXT_COL].apply(clean_text)
    # Remove rows where text became empty after cleaning
    df.drop(df[df[TEXT_COL] == ""].index, inplace=True)
    after = len(df)
    dropped = before - after
    if dropped > 0:
        print(f"{name}: dropped {dropped} null/empty rows ({before} → {after})")

print(f"\nAfter cleaning — Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")

In [ ]:
# --- Encode Labels ---
label_encoder = LabelEncoder()
label_encoder.fit(df_train[LABEL_COL])

NUM_CLASSES = len(label_encoder.classes_)
LABEL_NAMES = list(label_encoder.classes_)

df_train["label_id"] = label_encoder.transform(df_train[LABEL_COL])
df_val["label_id"]   = label_encoder.transform(df_val[LABEL_COL])
df_test["label_id"]  = label_encoder.transform(df_test[LABEL_COL])

print(f"Number of classes: {NUM_CLASSES}")
print(f"Label mapping:")
for idx, name in enumerate(LABEL_NAMES):
    print(f"  {idx} → {name}")

---
## 6. Tokenization

Each model loads its own pretrained tokenizer via `AutoTokenizer.from_pretrained(model_name)`. Even though all three backbones share the `bert-base-uncased` vocabulary, their tokenizer configurations differ (e.g. DistilBERT drops `token_type_ids`).

**Maximum sequence length is set to 128 tokens** (locked baseline from Phase 2 token stats: p95 = 119 tokens).

Tokenization is performed **without static padding** — dynamic padding via `DataCollatorWithPadding` is applied at collation time (see Section 7). Only `input_ids` and `attention_mask` are retained; `token_type_ids` is dropped.

In [ ]:
MAX_SEQ_LEN = 128

# Model identifiers — single source of truth
MODEL_IDS = {
    "minilm":     "nreimers/MiniLM-L6-H384-uncased",
    "distilbert": "distilbert-base-uncased",
    "bert":       "bert-base-uncased",
}

# Pre-load all tokenizers via AutoTokenizer
tokenizers = {
    key: AutoTokenizer.from_pretrained(model_id)
    for key, model_id in MODEL_IDS.items()
}

# Quick verification — tokenize WITHOUT padding (dynamic padding applied at collation)
sample_text = df_train[TEXT_COL].iloc[0][:200]
for name, tok in tokenizers.items():
    enc = tok(sample_text, max_length=MAX_SEQ_LEN, truncation=True)
    print(f"{name:12s} — input_ids length: {len(enc['input_ids'])}, "
          f"keys: {list(enc.keys())}")

---
## 7. Dataset Conversion — PyTorch Dataset & DataLoader with Dynamic Padding

Custom `Dataset` class wrapping the tokenized inputs for each transformer model. **Dynamic padding** via `DataCollatorWithPadding` pads each batch only to the longest sequence in that batch (not to `max_seq_len` globally), significantly reducing activation memory per batch on 6 GB VRAM.

In [ ]:
class WAFDataset(Dataset):
    """PyTorch Dataset for WAF HTTP request classification.
    
    Tokenizes without padding — dynamic padding is applied at collation time
    by DataCollatorWithPadding, padding each batch only to its longest sequence.
    """

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        # Tokenize WITHOUT padding — collator handles dynamic padding
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(label, dtype=torch.long),
        }
        return item


def build_dataloaders(tokenizer, batch_size, num_workers=2, pin_memory=True):
    """Create train/val/test DataLoaders with dynamic padding via DataCollatorWithPadding."""
    ds_train = WAFDataset(df_train[TEXT_COL], df_train["label_id"], tokenizer, MAX_SEQ_LEN)
    ds_val   = WAFDataset(df_val[TEXT_COL],   df_val["label_id"],   tokenizer, MAX_SEQ_LEN)
    ds_test  = WAFDataset(df_test[TEXT_COL],  df_test["label_id"],  tokenizer, MAX_SEQ_LEN)

    # DataCollatorWithPadding pads each batch to the longest sequence in that batch
    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

    loader_kwargs = dict(num_workers=num_workers, pin_memory=pin_memory, collate_fn=collator)

    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  **loader_kwargs)
    val_loader   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, **loader_kwargs)
    test_loader  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False, **loader_kwargs)

    return train_loader, val_loader, test_loader

print("WAFDataset and DataLoader builder (with DataCollatorWithPadding) ready.")

---
## 8. GPU Configuration — RTX 3060 Laptop (6 GB VRAM) Optimization

Training on a **RTX 3060 Laptop with 6 GB VRAM** requires per-model batch sizing. All three models target an **effective batch size of 128** for comparable optimization dynamics.

| Model | Per-Device Batch | Grad Accum Steps | Effective Batch | Gradient Checkpointing |
|-------|-----------------|-----------------|-----------------|------------------------|
| MiniLM-L6 | 16 | 8 | 128 | No |
| DistilBERT | 16 | 8 | 128 | No |
| BERT-base | 8 | 16 | 128 | Yes |

Additional strategies:
| Strategy | Setting |
|----------|--------|
| **Mixed precision** | FP16 via `torch.cuda.amp` (Ampere Tensor Core native) |
| **Sequence length** | 128 tokens (locked baseline) |
| **Gradient clipping** | `max_grad_norm = 1.0` |
| **Class-weighted loss** | Inverse-frequency weighted `nn.CrossEntropyLoss` |
| **Full fine-tuning** | All layers trainable (no freezing) |
| **Warmup** | Linear warmup over 6% of total steps |
| **Early stopping** | patience=3 on validation Macro-F1 |

In [ ]:
# ============================================================
# 8.1  Per-Model Training Configuration
# ============================================================

# Shared hyperparameters
BASE_CONFIG = {
    "learning_rate":    2e-5,
    "weight_decay":     0.01,
    "max_grad_norm":    1.0,
    "epochs":           5,
    "max_seq_length":   MAX_SEQ_LEN,   # 128
    "warmup_ratio":     0.06,          # 6% of total steps
    "early_stop_patience": 3,          # patience on val Macro-F1
    "fp16":             True,
    "num_workers":      2,
    "pin_memory":       True,
    "seed":             SEED,
}

# Per-model overrides (batch size × accum = 128 for all)
MODEL_CONFIGS = {
    "minilm": {
        **BASE_CONFIG,
        "model_id":                    MODEL_IDS["minilm"],
        "per_device_train_batch_size": 16,
        "gradient_accumulation_steps": 8,
        "gradient_checkpointing":      False,
    },
    "distilbert": {
        **BASE_CONFIG,
        "model_id":                    MODEL_IDS["distilbert"],
        "per_device_train_batch_size": 16,
        "gradient_accumulation_steps": 8,
        "gradient_checkpointing":      False,
    },
    "bert": {
        **BASE_CONFIG,
        "model_id":                    MODEL_IDS["bert"],
        "per_device_train_batch_size": 8,
        "gradient_accumulation_steps": 16,
        "gradient_checkpointing":      True,
    },
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Per-Model Training Configurations:")
for name, cfg in MODEL_CONFIGS.items():
    eff = cfg["per_device_train_batch_size"] * cfg["gradient_accumulation_steps"]
    print(f"\n  {name}:")
    print(f"    batch_size={cfg['per_device_train_batch_size']}, "
          f"accum={cfg['gradient_accumulation_steps']}, "
          f"effective={eff}, "
          f"grad_ckpt={cfg['gradient_checkpointing']}")
print(f"\n  Device: {DEVICE}")

---
## 9. VRAM Pre-Flight Check & Class Weights

### Pre-Flight VRAM Check
Before committing to a full training run, verify that each model fits in 6 GB VRAM by running a single forward+backward pass at the intended batch size and sequence length.

### Class-Weighted Loss
The training split has Code Injection at ~3.5% vs SQL Injection at ~46.7% — a ~13:1 ratio. Standard unweighted cross-entropy would largely ignore the minority class. Inverse-frequency weighted `nn.CrossEntropyLoss` corrects this:

$$w_c = \frac{N_{\text{total}}}{K \times N_c}$$

In [ ]:
# ============================================================
# 9.1  VRAM Pre-Flight Check
# ============================================================

def check_vram(model_name, model_id, batch_size, seq_len, device):
    """Run a single forward+backward pass and report peak VRAM usage."""
    if device.type != "cuda":
        print(f"  [{model_name}] Skipping VRAM check — no CUDA device")
        return 0.0

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_CLASSES)
    model.to(device)
    model.train()

    # Synthetic batch
    dummy_ids = torch.randint(0, 1000, (batch_size, seq_len), device=device)
    dummy_mask = torch.ones(batch_size, seq_len, dtype=torch.long, device=device)
    dummy_labels = torch.zeros(batch_size, dtype=torch.long, device=device)

    with autocast(device_type="cuda", enabled=True):
        outputs = model(input_ids=dummy_ids, attention_mask=dummy_mask, labels=dummy_labels)
        loss = outputs.loss

    loss.backward()

    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"  [{model_name}] batch_size={batch_size}, seq_len={seq_len} → "
          f"Peak VRAM: {peak_mb:.0f} MB / 6144 MB "
          f"({'✓ OK' if peak_mb < 5200 else '⚠ TIGHT'})")

    del model, dummy_ids, dummy_mask, dummy_labels, outputs, loss
    torch.cuda.empty_cache()
    return peak_mb

print("=== VRAM Pre-Flight Check ===")
for key, cfg in MODEL_CONFIGS.items():
    check_vram(
        model_name=key,
        model_id=cfg["model_id"],
        batch_size=cfg["per_device_train_batch_size"],
        seq_len=MAX_SEQ_LEN,
        device=DEVICE,
    )

# ============================================================
# 9.2  Compute Inverse-Frequency Class Weights
# ============================================================

def compute_class_weights(labels, num_classes, device):
    """Compute inverse-frequency class weights: w_c = N_total / (K * N_c)."""
    counts = np.bincount(labels, minlength=num_classes).astype(np.float64)
    total = counts.sum()
    weights = total / (num_classes * counts)
    weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
    return weights_tensor

CLASS_WEIGHTS = compute_class_weights(df_train["label_id"].values, NUM_CLASSES, DEVICE)

print("\nClass Weights (inverse-frequency):")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name:25s}: {CLASS_WEIGHTS[i].item():.4f}")

---
## 10. Training Pipeline

Generic training and validation routines used by all three models. Features:

- **AdamW optimizer** with two parameter groups (weight decay excluded from bias + LayerNorm)
- **Class-weighted CrossEntropyLoss** (inverse-frequency)
- **Gradient accumulation** (effective batch size = 128 for all models)
- **Gradient clipping** at `max_grad_norm = 1.0`
- **Mixed precision (FP16)** via `autocast` + `GradScaler`
- **Linear warmup** scheduler (6% of total steps)
- **Early stopping** with patience=3 on validation Macro-F1
- **Gradient checkpointing** for BERT-base
- Full fine-tuning with no frozen layers

In [ ]:
import time
import math

def build_optimizer(model, lr, weight_decay):
    """Build AdamW with two parameter groups: decay vs no-decay (bias, LayerNorm)."""
    no_decay = ["bias", "LayerNorm.weight"]
    grouped = [
        {
            "params": [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    return torch.optim.AdamW(grouped, lr=lr)


def train_one_epoch(model, dataloader, optimizer, scheduler, scaler, criterion,
                    device, accum_steps, max_grad_norm,
                    epoch=None, total_epochs=None, model_name="", log_every=500):
    """Train for one epoch with gradient accumulation, FP16, grad clipping, and warmup."""
    model.train()
    total_loss = 0.0
    running_loss = 0.0
    optimizer.zero_grad()
    total_steps = len(dataloader)
    epoch_start = time.time()

    epoch_tag = f"Epoch {epoch}/{total_epochs}" if epoch else "Epoch"
    print(f"\n  ▶ [{model_name}] {epoch_tag} — Training started ({total_steps:,} steps) ...")

    for step, batch in enumerate(dataloader):
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels) / accum_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * accum_steps
        running_loss += loss.item() * accum_steps

        if (step + 1) % accum_steps == 0 or (step + 1) == total_steps:
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if (step + 1) % log_every == 0 or (step + 1) == total_steps:
            elapsed = time.time() - epoch_start
            avg_loss = running_loss / (step + 1)
            pct = (step + 1) / total_steps * 100
            eta = elapsed / (step + 1) * (total_steps - step - 1)
            print(f"    Step {step+1:>6,}/{total_steps:,} ({pct:5.1f}%)  "
                  f"Loss: {avg_loss:.4f}  "
                  f"Elapsed: {elapsed:.0f}s  ETA: {eta:.0f}s")

    epoch_time = time.time() - epoch_start
    print(f"  ✓ [{model_name}] {epoch_tag} — Training done in {epoch_time:.0f}s")
    return total_loss / total_steps


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    """Evaluate model on a dataloader. Returns (loss, accuracy, all_preds, all_labels, all_logits)."""
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_logits = [], [], []

    for batch in dataloader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_logits.append(outputs.logits.float().cpu())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    all_logits = torch.cat(all_logits, dim=0)
    return avg_loss, acc, np.array(all_preds), np.array(all_labels), all_logits


def run_training(model_key, model_name_display):
    """
    Full training loop for a transformer model with:
    - Per-model config (batch size, gradient accumulation, gradient checkpointing)
    - Two-group AdamW (bias/LayerNorm excluded from weight decay)
    - Class-weighted CrossEntropyLoss
    - Linear warmup scheduler (6% of total steps)
    - Gradient clipping (max_grad_norm=1.0)
    - Early stopping on val Macro-F1 with patience=3
    - FP16 mixed precision
    
    Returns: model, history dict, train_loader, val_loader, test_loader
    """
    cfg = MODEL_CONFIGS[model_key]
    model_id = cfg["model_id"]
    batch_size = cfg["per_device_train_batch_size"]
    accum_steps = cfg["gradient_accumulation_steps"]
    num_epochs = cfg["epochs"]
    lr = cfg["learning_rate"]
    wd = cfg["weight_decay"]
    max_grad_norm = cfg["max_grad_norm"]
    warmup_ratio = cfg["warmup_ratio"]
    patience = cfg["early_stop_patience"]
    use_grad_ckpt = cfg["gradient_checkpointing"]

    # Load model
    print(f"Loading {model_id} ...")
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_CLASSES)

    # Enable gradient checkpointing if configured
    if use_grad_ckpt:
        model.gradient_checkpointing_enable()
        print(f"  Gradient checkpointing ENABLED for {model_name_display}")

    model.to(DEVICE)

    # Build DataLoaders
    tok = tokenizers[model_key]
    train_loader, val_loader, test_loader = build_dataloaders(
        tok, batch_size, cfg["num_workers"], cfg["pin_memory"]
    )

    # Optimizer with two parameter groups
    optimizer = build_optimizer(model, lr, wd)

    # Scheduler: linear warmup
    steps_per_epoch = math.ceil(len(train_loader) / accum_steps)
    total_training_steps = steps_per_epoch * num_epochs
    warmup_steps = int(warmup_ratio * total_training_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps,
    )

    # FP16 scaler
    scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

    # Class-weighted loss
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

    # Tracking
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    effective_bs = batch_size * accum_steps

    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}
    best_val_f1 = 0.0
    best_state = None
    no_improve_count = 0

    print(f"\n{'='*65}")
    print(f" Training: {model_name_display}")
    print(f" Epochs: {num_epochs} (early stop patience={patience}) | LR: {lr}")
    print(f" Batch: {batch_size} × {accum_steps} = {effective_bs} effective")
    print(f" Warmup: {warmup_steps}/{total_training_steps} steps ({warmup_ratio*100:.0f}%)")
    print(f" Grad clip: {max_grad_norm} | Grad ckpt: {use_grad_ckpt}")
    print(f" Total params: {total_params:,} | Trainable: {trainable_params:,} (100%)")
    print(f"{'='*65}")

    training_start = time.time()

    for epoch in range(1, num_epochs + 1):
        # Train
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scheduler, scaler, criterion, DEVICE,
            accum_steps, max_grad_norm,
            epoch=epoch, total_epochs=num_epochs,
            model_name=model_name_display, log_every=500
        )
        # Validate
        val_loss, val_acc, val_preds, val_labels, _ = evaluate(model, val_loader, criterion, DEVICE)
        val_f1 = f1_score(val_labels, val_preds, average="macro")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        # Log VRAM if available
        vram_str = ""
        if DEVICE.type == "cuda":
            peak_mb = torch.cuda.max_memory_allocated() / 1e6
            vram_str = f"  VRAM: {peak_mb:.0f}MB"

        print(f"  Epoch {epoch}/{num_epochs}  "
              f"Train Loss: {train_loss:.4f}  "
              f"Val Loss: {val_loss:.4f}  "
              f"Val Acc: {val_acc:.4f}  "
              f"Val Macro-F1: {val_f1:.4f}{vram_str}")

        # Early stopping on Macro-F1
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve_count = 0
            print(f"    ↑ New best Val Macro-F1: {best_val_f1:.4f} — checkpoint saved")
        else:
            no_improve_count += 1
            print(f"    → No improvement ({no_improve_count}/{patience})")
            if no_improve_count >= patience:
                print(f"  ■ Early stopping triggered at epoch {epoch}")
                break

    total_training_time = time.time() - training_start

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(DEVICE)
        print(f"\n  ✓ Restored best checkpoint (Val Macro-F1: {best_val_f1:.4f})")

    history["total_training_time_s"] = total_training_time
    history["total_params"] = total_params
    history["trainable_params"] = trainable_params
    history["best_val_f1"] = best_val_f1

    print(f"  Total training time: {total_training_time:.0f}s "
          f"({total_training_time/60:.1f} min)")

    return model, history, train_loader, val_loader, test_loader

print("Training pipeline functions defined.")

---
## 11. Training Execution

Execution order follows the implementation plan: **MiniLM-L6 → DistilBERT → BERT-base**. MiniLM is trained first as a fast pipeline validation (~10 min/epoch). BERT-base runs last with gradient checkpointing enabled.

Each model section:
1. Train with per-model config → early stopping on Macro-F1
2. Evaluate on held-out test set

---
### 11.1 MODEL 1 — MiniLM-L6 (`nreimers/MiniLM-L6-H384-uncased`)

MiniLM-L6 is the smallest model (~22.7M params, 6 layers, 384-dim). Trained first to validate the complete pipeline end-to-end before committing hours to larger models.

In [ ]:
# --- MiniLM: Train ---
minilm_model, minilm_history, minilm_train_loader, minilm_val_loader, minilm_test_loader = \
    run_training("minilm", "MiniLM-L6")

In [ ]:
# --- MiniLM: Test-Set Evaluation ---
criterion_eval = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
minilm_test_loss, minilm_test_acc, minilm_preds, minilm_labels, minilm_test_logits = evaluate(
    minilm_model, minilm_test_loader, criterion_eval, DEVICE
)
print(f"MiniLM — Test Accuracy: {minilm_test_acc:.4f}")
print(f"\nClassification Report:\n")
print(classification_report(minilm_labels, minilm_preds, target_names=LABEL_NAMES, digits=4))

---
### 11.2 MODEL 2 — DistilBERT (`distilbert-base-uncased`)

DistilBERT is a distilled version of BERT with 6 transformer layers and ~66M parameters. Trained with batch_size=16, accum=8, no gradient checkpointing.

In [ ]:
# --- DistilBERT: Train ---
distilbert_model, distilbert_history, distilbert_train_loader, distilbert_val_loader, distilbert_test_loader = \
    run_training("distilbert", "DistilBERT")

In [ ]:
# --- DistilBERT: Test-Set Evaluation ---
distilbert_test_loss, distilbert_test_acc, distilbert_preds, distilbert_labels, distilbert_test_logits = evaluate(
    distilbert_model, distilbert_test_loader, criterion_eval, DEVICE
)
print(f"DistilBERT — Test Accuracy: {distilbert_test_acc:.4f}")
print(f"\nClassification Report:\n")
print(classification_report(distilbert_labels, distilbert_preds, target_names=LABEL_NAMES, digits=4))

---
### 11.3 MODEL 3 — BERT (`bert-base-uncased`)

BERT (12 transformer encoder layers, 110M parameters). Trained with batch_size=8, accum=16, **gradient checkpointing enabled** due to VRAM constraints.

In [ ]:
# --- BERT: Train ---
bert_model, bert_history, bert_train_loader, bert_val_loader, bert_test_loader = \
    run_training("bert", "BERT")

In [ ]:
# --- BERT: Test-Set Evaluation ---
bert_test_loss, bert_test_acc, bert_preds, bert_labels, bert_test_logits = evaluate(
    bert_model, bert_test_loader, criterion_eval, DEVICE
)
print(f"BERT — Test Accuracy: {bert_test_acc:.4f}")
print(f"\nClassification Report:\n")
print(classification_report(bert_labels, bert_preds, target_names=LABEL_NAMES, digits=4))

---
## 12. Temperature Scaling Calibration

Raw softmax outputs from fine-tuned transformers are systematically overconfident. Since the confidence thresholds (LOW <50%, MEDIUM 50–80%, HIGH >80%) drive real enforcement actions, **calibrated probabilities** are required.

Temperature scaling (Guo et al., 2017) fits a single scalar $T$ on the validation set by minimizing NLL, then applies: $\text{calibrated\_logits} = \text{logits} / T$

Expected Calibration Error (ECE) is computed before and after calibration.

In [ ]:
# ============================================================
# 12.1  Temperature Scaling Implementation
# ============================================================

def compute_ece(probs, labels, n_bins=10):
    """Compute Expected Calibration Error (ECE) with n_bins."""
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels).astype(float)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += mask.sum() / len(labels) * abs(bin_acc - bin_conf)
    return ece


def fit_temperature(val_logits, val_labels):
    """Fit temperature T on validation set by minimizing NLL."""
    val_logits_t = torch.tensor(val_logits, dtype=torch.float32) if not isinstance(val_logits, torch.Tensor) else val_logits.float()
    val_labels_t = torch.tensor(val_labels, dtype=torch.long) if not isinstance(val_labels, torch.Tensor) else val_labels

    def nll_with_temp(T):
        scaled = val_logits_t / T
        log_probs = torch.log_softmax(scaled, dim=-1)
        nll = nn.functional.nll_loss(log_probs, val_labels_t).item()
        return nll

    result = minimize_scalar(nll_with_temp, bounds=(0.1, 10.0), method="bounded")
    return result.x


def apply_temperature(logits, T):
    """Apply temperature scaling to logits and return calibrated probabilities."""
    if isinstance(logits, np.ndarray):
        logits = torch.tensor(logits, dtype=torch.float32)
    scaled = logits / T
    return torch.softmax(scaled, dim=-1).numpy()


# Fit temperature for each model using validation logits
calibration_results = {}

for model_key, model_name, model_obj, val_loader in [
    ("minilm", "MiniLM", minilm_model, minilm_val_loader),
    ("distilbert", "DistilBERT", distilbert_model, distilbert_val_loader),
    ("bert", "BERT", bert_model, bert_val_loader),
]:
    # Get validation logits
    _, _, _, val_labels_arr, val_logits = evaluate(model_obj, val_loader, criterion_eval, DEVICE)

    # Fit temperature
    T = fit_temperature(val_logits, val_labels_arr)

    # Compute ECE before and after
    probs_before = torch.softmax(val_logits.float(), dim=-1).numpy()
    probs_after = apply_temperature(val_logits, T)

    ece_before = compute_ece(probs_before, val_labels_arr)
    ece_after = compute_ece(probs_after, val_labels_arr)

    calibration_results[model_key] = {
        "temperature": T,
        "ece_before": ece_before,
        "ece_after": ece_after,
    }

    print(f"{model_name:12s} — T = {T:.4f}  |  ECE: {ece_before:.4f} → {ece_after:.4f}")

print("\nCalibration complete.")

In [ ]:
# ============================================================
# 12.2  Reliability Diagrams (10-bin)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model_key, model_name, test_logits) in zip(axes, [
    ("minilm", "MiniLM", minilm_test_logits),
    ("distilbert", "DistilBERT", distilbert_test_logits),
    ("bert", "BERT", bert_test_logits),
]):
    T = calibration_results[model_key]["temperature"]
    test_labels = {"minilm": minilm_labels, "distilbert": distilbert_labels, "bert": bert_labels}[model_key]

    probs_raw = torch.softmax(test_logits.float(), dim=-1).numpy()
    probs_cal = apply_temperature(test_logits, T)

    n_bins = 10
    bin_boundaries = np.linspace(0, 1, n_bins + 1)

    for probs, label, color, ls in [
        (probs_raw, "Before (raw)", "#F44336", "--"),
        (probs_cal, f"After (T={T:.2f})", "#4CAF50", "-"),
    ]:
        confs = np.max(probs, axis=1)
        preds = np.argmax(probs, axis=1)
        accs = (preds == test_labels).astype(float)
        bin_accs, bin_confs = [], []
        for i in range(n_bins):
            mask = (confs > bin_boundaries[i]) & (confs <= bin_boundaries[i + 1])
            if mask.sum() > 0:
                bin_accs.append(accs[mask].mean())
                bin_confs.append(confs[mask].mean())
        ax.plot(bin_confs, bin_accs, f"o{ls}", color=color, label=label, linewidth=2)

    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title(f"{model_name} — Reliability Diagram", fontsize=12, fontweight="bold")
    ax.set_xlabel("Mean Predicted Confidence")
    ax.set_ylabel("Fraction of Positives (Accuracy)")
    ax.legend(fontsize=9)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig("reliability_diagrams.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reliability_diagrams.png")

---
## 13. Model Evaluation — Comprehensive Thesis Metrics

Compute a full suite of evaluation metrics on the **test set** for all three models, covering all standard metrics required for academic reporting:

| Category | Metrics |
|---|---|
| **Overall** | Accuracy, Macro F1, Weighted F1, Cohen's Kappa, Matthews Correlation Coefficient (MCC), Log Loss |
| **Per-class** | Precision, Recall, F1-Score, Support |
| **Probabilistic** | ROC-AUC (OvR), Average Precision (PR-AUC) |
| **Ranking** | Per-class ROC curves, Precision-Recall curves |
| **Agreement** | Confusion Matrix (raw + normalized) |

In [ ]:
# =====================================================================
# 13.1  Collect CALIBRATED softmax probabilities for evaluation
# =====================================================================

# Apply temperature scaling to test logits for each model
bert_probs = apply_temperature(bert_test_logits, calibration_results["bert"]["temperature"])
distilbert_probs = apply_temperature(distilbert_test_logits, calibration_results["distilbert"]["temperature"])
minilm_probs = apply_temperature(minilm_test_logits, calibration_results["minilm"]["temperature"])

print("Calibrated probabilities computed for all models.")

# =====================================================================
# 13.2  Comprehensive metrics computation
# =====================================================================

def compute_full_metrics(y_true, y_pred, y_probs, model_name, history, label_names):
    """
    Compute all thesis-relevant metrics for one model.
    Returns a dict with scalar metrics and stores per-class detail.
    """
    n_classes = len(label_names)
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

    # --- Overall metrics ---
    acc       = accuracy_score(y_true, y_pred)
    macro_p   = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_r   = recall_score(y_true, y_pred, average="macro", zero_division=0)
    macro_f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    weighted_p  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    weighted_r  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    mcc       = matthews_corrcoef(y_true, y_pred)
    kappa     = cohen_kappa_score(y_true, y_pred)
    logloss   = log_loss(y_true, y_probs, labels=list(range(n_classes)))

    # --- FPR (macro average) ---
    cm = confusion_matrix(y_true, y_pred)
    fprs = []
    for c in range(n_classes):
        fp = cm[:, c].sum() - cm[c, c]
        tn = cm.sum() - cm[c, :].sum() - cm[:, c].sum() + cm[c, c]
        fprs.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
    macro_fpr = np.mean(fprs)

    # --- ROC-AUC (One-vs-Rest) ---
    try:
        roc_auc_macro = roc_auc_score(y_true_bin, y_probs, average="macro", multi_class="ovr")
        roc_auc_weighted = roc_auc_score(y_true_bin, y_probs, average="weighted", multi_class="ovr")
    except ValueError:
        roc_auc_macro = roc_auc_weighted = float("nan")

    # --- Average Precision (PR-AUC) ---
    try:
        ap_macro = average_precision_score(y_true_bin, y_probs, average="macro")
        ap_weighted = average_precision_score(y_true_bin, y_probs, average="weighted")
    except ValueError:
        ap_macro = ap_weighted = float("nan")

    # --- Per-class ROC-AUC ---
    per_class_roc = {}
    for i, name in enumerate(label_names):
        try:
            per_class_roc[name] = roc_auc_score(y_true_bin[:, i], y_probs[:, i])
        except ValueError:
            per_class_roc[name] = float("nan")

    # --- Training metadata ---
    train_time = history.get("total_training_time_s", 0)
    total_p = history.get("total_params", 0)
    trainable_p = history.get("trainable_params", 0)

    return {
        "Model":               model_name,
        "Accuracy":            acc,
        "Macro Precision":     macro_p,
        "Macro Recall":        macro_r,
        "Macro F1":            macro_f1,
        "Macro FPR":           macro_fpr,
        "Weighted F1":         weighted_f1,
        "Weighted Precision":  weighted_p,
        "Weighted Recall":     weighted_r,
        "MCC":                 mcc,
        "Cohen's Kappa":       kappa,
        "Log Loss":            logloss,
        "ROC-AUC (Macro)":     roc_auc_macro,
        "ROC-AUC (Weighted)":  roc_auc_weighted,
        "PR-AUC (Macro)":      ap_macro,
        "PR-AUC (Weighted)":   ap_weighted,
        "Per-class ROC-AUC":   per_class_roc,
        "Training Time (s)":   round(train_time, 1),
        "Total Params":        total_p,
        "Trainable Params":    trainable_p,
    }

results = [
    compute_full_metrics(bert_labels,       bert_preds,       bert_probs,       "BERT",       bert_history,       LABEL_NAMES),
    compute_full_metrics(distilbert_labels, distilbert_preds, distilbert_probs, "DistilBERT", distilbert_history, LABEL_NAMES),
    compute_full_metrics(minilm_labels,     minilm_preds,     minilm_probs,     "MiniLM",     minilm_history,     LABEL_NAMES),
]

# --- Print summary ---
for r in results:
    print(f"\n{'='*55}")
    print(f" {r['Model']}")
    print(f"{'='*55}")
    for k, v in r.items():
        if k in ("Model", "Per-class ROC-AUC"):
            continue
        if isinstance(v, float):
            print(f"  {k:25s}: {v:.4f}")
        else:
            print(f"  {k:25s}: {v:,}")
    print(f"  {'Per-class ROC-AUC':25s}:")
    for cls_name, auc_val in r["Per-class ROC-AUC"].items():
        print(f"    {cls_name:23s}: {auc_val:.4f}")

---
## 14. Model Comparison Table

In [ ]:
# =====================================================================
# 14.  Model Comparison Table  (thesis-ready)
# =====================================================================

# Build a clean DataFrame  — drop the nested dict column
scalar_keys = [k for k in results[0] if k != "Per-class ROC-AUC"]
comparison_df = pd.DataFrame([{k: r[k] for k in scalar_keys} for r in results])
comparison_df = comparison_df.set_index("Model")

# Format numeric columns for display
fmt = {}
for col in comparison_df.columns:
    if "Param" in col:
        fmt[col] = "{:,.0f}"
    elif "Time" in col:
        fmt[col] = "{:.1f}"
    else:
        fmt[col] = "{:.4f}"

# Columns where higher is better
higher_better = [
    c for c in comparison_df.columns
    if c not in ("Log Loss", "Macro FPR", "Training Time (s)", "Total Params", "Trainable Params")
]
# Columns where lower is better
lower_better = [c for c in comparison_df.columns if c in ("Log Loss", "Macro FPR")]

styled = (
    comparison_df.style
    .format(fmt)
    .highlight_max(axis=0, color="#c6efce", subset=higher_better)
    .highlight_min(axis=0, color="#c6efce", subset=lower_better)
    .set_caption("Comprehensive Test-Set Evaluation — All Models (Calibrated Probabilities)")
)
display(styled)

print("\nPlain-text version for copying into LaTeX / Word:")
print(comparison_df.to_string())

In [ ]:
# =====================================================================
# 14.1  Per-Class Metrics Table  (Precision / Recall / F1 per class per model)
# =====================================================================

preds_map  = {"BERT": bert_preds,  "DistilBERT": distilbert_preds,  "MiniLM": minilm_preds}
labels_map = {"BERT": bert_labels, "DistilBERT": distilbert_labels, "MiniLM": minilm_labels}

rows = []
for r in results:
    name = r["Model"]
    cr = classification_report(
        labels_map[name], preds_map[name],
        target_names=LABEL_NAMES, output_dict=True, zero_division=0,
    )
    for cls_name in LABEL_NAMES:
        rows.append({
            "Model": name,
            "Class": cls_name,
            "Precision": cr[cls_name]["precision"],
            "Recall":    cr[cls_name]["recall"],
            "F1-Score":  cr[cls_name]["f1-score"],
            "Support":   int(cr[cls_name]["support"]),
            "ROC-AUC":   r["Per-class ROC-AUC"].get(cls_name, float("nan")),
        })

perclass_df = pd.DataFrame(rows)

# Pretty-print per model
print("Per-Class Metrics by Model\n")
for model_name in ["BERT", "DistilBERT", "MiniLM"]:
    sub = perclass_df[perclass_df["Model"] == model_name].drop(columns="Model")
    sub = sub.set_index("Class")
    print(f"--- {model_name} ---")
    print(sub.to_string(float_format=lambda x: f"{x:.4f}"))
    print()

# Combined styled view
pivot = perclass_df.set_index(["Model", "Class"])
styled_pc = (
    pivot.style
    .format({c: "{:.4f}" for c in pivot.columns if c != "Support"})
    .format({"Support": "{:,.0f}"})
    .set_caption("Per-Class Test-Set Metrics — All Models")
)
display(styled_pc)

### 14.2 ROC Curves (One-vs-Rest)

Receiver Operating Characteristic curves plot $\text{TPR}$ against $\text{FPR}$ at varying decision thresholds.
A model with **AUC close to 1.0** demonstrates strong discriminative ability across all classes.

In [ ]:
# =====================================================================
# 14.2  ROC Curves — One-vs-Rest per model
# =====================================================================

probs_map  = {"BERT": bert_probs, "DistilBERT": distilbert_probs, "MiniLM": minilm_probs}
class_colors = ["#2196F3", "#F44336", "#FF9800", "#4CAF50"]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, model_name in zip(axes, ["BERT", "DistilBERT", "MiniLM"]):
    y_true = labels_map[model_name]
    y_prob = probs_map[model_name]
    y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

    for i, cls_name in enumerate(LABEL_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=class_colors[i], linewidth=2,
                label=f"{cls_name} (AUC={roc_auc_val:.3f})")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, linewidth=1)
    ax.set_title(f"{model_name} — ROC Curves (OvR)", fontsize=12, fontweight="bold")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: roc_curves.png")

### 14.3 Precision-Recall Curves

Precision-Recall curves are more informative than ROC curves for **imbalanced classes** (e.g., Code Injection at only 3.4%).
Average Precision (AP) summarises the curve as the weighted mean of precisions at each threshold.

In [ ]:
# =====================================================================
# 14.3  Precision-Recall Curves — One-vs-Rest per model
# =====================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, model_name in zip(axes, ["BERT", "DistilBERT", "MiniLM"]):
    y_true = labels_map[model_name]
    y_prob = probs_map[model_name]
    y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

    for i, cls_name in enumerate(LABEL_NAMES):
        prec, rec, _ = precision_recall_curve(y_true_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_prob[:, i])
        ax.plot(rec, prec, color=class_colors[i], linewidth=2,
                label=f"{cls_name} (AP={ap:.3f})")

    ax.set_title(f"{model_name} — Precision-Recall Curves", fontsize=12, fontweight="bold")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(fontsize=8, loc="lower left")
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.05])

plt.tight_layout()
plt.savefig("pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pr_curves.png")

---
## 15. Calibrated Confidence Scoring

After temperature scaling, confidence scores are **calibrated probabilities** — a stated 80% output means the model is correct approximately 80% of the time. Thresholds are locked:

- **LOW** confidence: < 50%
- **MEDIUM** confidence: 50–80%
- **HIGH** confidence: > 80%

$$P_{\text{calibrated}}(y=k \mid x) = \frac{e^{z_k/T}}{\sum_{j=1}^{K} e^{z_j/T}}$$

Where $T$ is the fitted temperature parameter.

In [ ]:
@torch.no_grad()
def get_calibrated_confidence(model, dataloader, device, temperature):
    """Get temperature-scaled calibrated probabilities and confidence scores."""
    model.eval()
    all_logits = []
    all_labels = []

    for batch in dataloader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        all_logits.append(outputs.logits.float().cpu())
        all_labels.extend(labels.cpu().numpy())

    all_logits = torch.cat(all_logits, dim=0)
    # Apply temperature scaling
    all_probs = apply_temperature(all_logits, temperature)
    confidence = np.max(all_probs, axis=1)
    predicted_classes = np.argmax(all_probs, axis=1)

    return all_probs, confidence, predicted_classes, np.array(all_labels)


# Compute calibrated confidence scores for the best model (select by highest Macro F1)
best_model_name = comparison_df["Macro F1"].idxmax()
print(f"Best model by Macro F1: {best_model_name}")

best_model_map = {
    "BERT":       (bert_model, bert_test_loader, calibration_results["bert"]["temperature"]),
    "DistilBERT": (distilbert_model, distilbert_test_loader, calibration_results["distilbert"]["temperature"]),
    "MiniLM":     (minilm_model, minilm_test_loader, calibration_results["minilm"]["temperature"]),
}

best_model_obj, best_test_loader, best_T = best_model_map[best_model_name]

probs, confidence, pred_classes, true_labels = get_calibrated_confidence(
    best_model_obj, best_test_loader, DEVICE, best_T
)

print(f"\nCalibrated Confidence Score Statistics ({best_model_name}, T={best_T:.4f}):")
print(f"  Mean   : {confidence.mean():.4f}")
print(f"  Median : {np.median(confidence):.4f}")
print(f"  Min    : {confidence.min():.4f}")
print(f"  Max    : {confidence.max():.4f}")

# Triage thresholds (from implementation plan — locked, never change)
LOW_THRESHOLD = 0.50
HIGH_THRESHOLD = 0.80

low_count = np.sum(confidence < LOW_THRESHOLD)
med_count = np.sum((confidence >= LOW_THRESHOLD) & (confidence < HIGH_THRESHOLD))
high_count = np.sum(confidence >= HIGH_THRESHOLD)

print(f"\nTriage Distribution (calibrated):")
print(f"  LOW  (< {LOW_THRESHOLD})    : {low_count:,} ({low_count/len(confidence)*100:.1f}%)")
print(f"  MED  ({LOW_THRESHOLD}–{HIGH_THRESHOLD})  : {med_count:,} ({med_count/len(confidence)*100:.1f}%)")
print(f"  HIGH (≥ {HIGH_THRESHOLD})    : {high_count:,} ({high_count/len(confidence)*100:.1f}%)")

---
## 16. Visualization

### 16.1 Training & Validation Loss Curves

In [ ]:
# --- Loss Curves ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

histories = {
    "BERT":       bert_history,
    "DistilBERT": distilbert_history,
    "MiniLM":     minilm_history,
}

for ax, (name, hist) in zip(axes, histories.items()):
    epochs_range = range(1, len(hist["train_loss"]) + 1)
    ax.plot(epochs_range, hist["train_loss"], "o-", label="Train Loss", color="#2196F3", linewidth=2)
    ax.plot(epochs_range, hist["val_loss"],   "s-", label="Val Loss",   color="#F44336", linewidth=2)
    ax.set_title(f"{name} — Loss Curves", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.set_xticks(list(epochs_range))

plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: loss_curves.png")

### 16.1b Validation Accuracy & F1 Curves

Track generalisation performance across epochs: validation accuracy and macro-F1.

In [ ]:
# --- Validation Accuracy & F1 Curves ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, hist) in zip(axes, histories.items()):
    epochs_range = range(1, len(hist["val_acc"]) + 1)
    ax.plot(epochs_range, hist["val_acc"], "o-", label="Val Accuracy", color="#4CAF50", linewidth=2)
    if "val_f1" in hist:
        ax.plot(epochs_range, hist["val_f1"], "s-", label="Val Macro F1", color="#9C27B0", linewidth=2)
    ax.set_title(f"{name} — Validation Metrics", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    ax.legend()
    ax.set_xticks(list(epochs_range))
    ax.set_ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig("val_accuracy_f1_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: val_accuracy_f1_curves.png")

### 16.2 Confusion Matrix Heatmaps

In [ ]:
# --- Confusion Matrices: Raw counts (top row) & Normalized % (bottom row) ---
all_predictions = {
    "BERT":       (bert_labels,       bert_preds),
    "DistilBERT": (distilbert_labels, distilbert_preds),
    "MiniLM":     (minilm_labels,     minilm_preds),
}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for col, (name, (y_true, y_pred)) in enumerate(all_predictions.items()):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

    # Row 0: Raw counts
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
        ax=axes[0, col], linewidths=0.5, linecolor="gray",
    )
    axes[0, col].set_title(f"{name} — Raw Counts", fontsize=12, fontweight="bold")
    axes[0, col].set_xlabel("Predicted")
    axes[0, col].set_ylabel("Actual")
    axes[0, col].tick_params(axis="x", rotation=30)
    axes[0, col].tick_params(axis="y", rotation=0)

    # Row 1: Normalised (%)
    sns.heatmap(
        cm_norm, annot=True, fmt=".2%", cmap="YlOrRd",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
        ax=axes[1, col], linewidths=0.5, linecolor="gray",
        vmin=0, vmax=1,
    )
    axes[1, col].set_title(f"{name} — Normalised (%)", fontsize=12, fontweight="bold")
    axes[1, col].set_xlabel("Predicted")
    axes[1, col].set_ylabel("Actual")
    axes[1, col].tick_params(axis="x", rotation=30)
    axes[1, col].tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices.png")

### 16.3 Confidence Score Distribution

In [ ]:
# --- Confidence Distribution Histogram ---
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(confidence, bins=50, color="#4CAF50", edgecolor="black", alpha=0.8)
ax.axvline(LOW_THRESHOLD, color="orange", linestyle="--", linewidth=2, label=f"Low threshold ({LOW_THRESHOLD})")
ax.axvline(HIGH_THRESHOLD, color="red", linestyle="--", linewidth=2, label=f"High threshold ({HIGH_THRESHOLD})")
ax.set_title(f"{best_model_name} — Confidence Score Distribution (Test Set)", fontsize=13, fontweight="bold")
ax.set_xlabel("Confidence Score")
ax.set_ylabel("Count")
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confidence_distribution.png")

### 16.4 Model Comparison Bar Chart

In [ ]:
# --- Model Comparison Bar Chart (extended) ---
metrics_to_plot = ["Accuracy", "Macro F1", "Weighted F1", "MCC", "Cohen's Kappa", "ROC-AUC (Macro)"]
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))

model_colors = {"BERT": "#2196F3", "DistilBERT": "#FF9800", "MiniLM": "#4CAF50"}

for i, r in enumerate(results):
    values = [r[m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, values, width, label=r["Model"],
                  color=model_colors[r["Model"]], edgecolor="black", linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot, fontsize=10, rotation=15)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("Model Comparison — Test Set Metrics", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: model_comparison.png")

---
## 17. Save Versioned Artifacts

Each model's artifacts are saved to `model_registry/staging/{model_name}_{dataset_version}_{timestamp}/` containing checkpoint, training log, evaluation report, calibration data, config, and git hash.

In [ ]:
DATASET_VERSION = "SRBH_clean_v3.1.0"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
REGISTRY_BASE = os.path.join("..", "model_registry", "staging")

# Get git hash for reproducibility
try:
    git_hash = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL
    ).decode().strip()
except Exception:
    git_hash = "unknown"

models_to_save = {
    "minilm":     (minilm_model,     tokenizers["minilm"],     "MiniLM",     minilm_history),
    "distilbert": (distilbert_model, tokenizers["distilbert"], "DistilBERT", distilbert_history),
    "bert":       (bert_model,       tokenizers["bert"],       "BERT",       bert_history),
}

for key, (model, tokenizer, display_name, history) in models_to_save.items():
    run_dir = os.path.join(REGISTRY_BASE, f"{key}_{DATASET_VERSION}_{TIMESTAMP}")

    # Create directories
    ckpt_dir = os.path.join(run_dir, "checkpoint")
    cal_dir = os.path.join(run_dir, "calibration")
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(cal_dir, exist_ok=True)

    # Save model + tokenizer (HuggingFace format)
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    print(f"[{display_name}] Saved checkpoint → {ckpt_dir}")

    # Save training log
    training_log = {
        "model": display_name,
        "model_id": MODEL_IDS[key],
        "dataset_version": DATASET_VERSION,
        "seed": SEED,
        "epochs_trained": len(history["train_loss"]),
        "best_val_f1": history.get("best_val_f1"),
        "per_epoch": {
            "train_loss": history["train_loss"],
            "val_loss": history["val_loss"],
            "val_acc": history["val_acc"],
            "val_f1": history["val_f1"],
        },
        "total_training_time_s": history.get("total_training_time_s"),
        "total_params": history.get("total_params"),
        "trainable_params": history.get("trainable_params"),
    }
    with open(os.path.join(run_dir, "training_log.json"), "w") as f:
        json.dump(training_log, f, indent=2, default=str)

    # Save eval report
    test_labels_map = {"minilm": minilm_labels, "distilbert": distilbert_labels, "bert": bert_labels}
    test_preds_map = {"minilm": minilm_preds, "distilbert": distilbert_preds, "bert": bert_preds}

    cr = classification_report(
        test_labels_map[key], test_preds_map[key],
        target_names=LABEL_NAMES, output_dict=True, zero_division=0,
    )
    cm = confusion_matrix(test_labels_map[key], test_preds_map[key]).tolist()

    # Compute FPR
    r_match = [r for r in results if r["Model"] == display_name]
    macro_fpr = r_match[0]["Macro FPR"] if r_match else 0.0

    eval_report = {
        "per_class": cr,
        "confusion_matrix": cm,
        "accuracy": accuracy_score(test_labels_map[key], test_preds_map[key]),
        "macro_f1": f1_score(test_labels_map[key], test_preds_map[key], average="macro"),
        "macro_fpr": macro_fpr,
    }
    with open(os.path.join(run_dir, "eval_report.json"), "w") as f:
        json.dump(eval_report, f, indent=2, default=str)

    # Save calibration data
    cal_data = {
        "temperature": calibration_results[key]["temperature"],
        "ece_before": calibration_results[key]["ece_before"],
        "ece_after": calibration_results[key]["ece_after"],
    }
    with open(os.path.join(cal_dir, "calibration.json"), "w") as f:
        json.dump(cal_data, f, indent=2)

    # Save config used (as JSON since yaml may not be installed)
    with open(os.path.join(run_dir, "config_used.json"), "w") as f:
        json.dump(MODEL_CONFIGS[key], f, indent=2, default=str)

    # Save git hash
    with open(os.path.join(run_dir, "git_hash.txt"), "w") as f:
        f.write(git_hash + "\n")

    print(f"[{display_name}] Artifacts saved → {run_dir}")

# Save label map (shared)
label_map = {int(i): lbl for i, lbl in enumerate(LABEL_NAMES)}
label_map_path = os.path.join(REGISTRY_BASE, "label_map.json")
with open(label_map_path, "w") as f:
    json.dump(label_map, f, indent=2)
print(f"\nSaved label map: {label_map_path}")

# Save comparison results
comparison_df.to_csv(os.path.join(REGISTRY_BASE, f"model_comparison_{TIMESTAMP}.csv"))
print(f"Saved comparison: model_comparison_{TIMESTAMP}.csv")

---
## 18. CPU Inference Latency Measurement

The production deployment is CPU inference on the same Ryzen 7 6800H. Target: **< 100ms mean**. Measure from saved checkpoint (not in-memory) to simulate cold-start inference.

In [ ]:
# ============================================================
# 18.1  CPU Inference Latency per Model
# ============================================================

cpu_device = torch.device("cpu")
latency_results = {}

# Use a representative sample payload
sample_payload = "GET /search?q=SELECT+*+FROM+users+WHERE+id%3D1+OR+1%3D1--"

for key, model_id in MODEL_IDS.items():
    display_name = {"minilm": "MiniLM", "distilbert": "DistilBERT", "bert": "BERT"}[key]

    # Load from checkpoint (simulates cold-start)
    run_dir = os.path.join(REGISTRY_BASE, f"{key}_{DATASET_VERSION}_{TIMESTAMP}", "checkpoint")
    cpu_model = AutoModelForSequenceClassification.from_pretrained(run_dir)
    cpu_tok = AutoTokenizer.from_pretrained(run_dir)
    cpu_model.to(cpu_device)
    cpu_model.eval()

    # Prepare input
    inputs = cpu_tok(
        sample_payload,
        max_length=MAX_SEQ_LEN,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    # Warmup (10 forward passes)
    for _ in range(10):
        with torch.no_grad():
            _ = cpu_model(**inputs)

    # Timed measurement (100 forward passes)
    times = []
    for _ in range(100):
        start = time.perf_counter()
        with torch.no_grad():
            _ = cpu_model(**inputs)
        times.append(time.perf_counter() - start)

    mean_ms = np.mean(times) * 1000
    p95_ms = np.percentile(times, 95) * 1000
    p99_ms = np.percentile(times, 99) * 1000

    latency_results[key] = {"mean_ms": mean_ms, "p95_ms": p95_ms, "p99_ms": p99_ms}

    status = "✓ PASS" if mean_ms < 100 else "✗ FAIL (consider ONNX export)"
    print(f"  {display_name:12s} — Mean: {mean_ms:.1f}ms | p95: {p95_ms:.1f}ms | p99: {p99_ms:.1f}ms  [{status}]")

    del cpu_model

print("\nAll latency measurements complete.")

---
## 19. Conclusion

### Summary

This notebook trained and evaluated three transformer-based classifiers for the WAF attack detection pipeline, following the Training Implementation Plan v2.0.

| Model | Architecture | Parameters | Fine-Tuning | Gradient Checkpointing |
|---|---|---|---|---|
| **MiniLM-L6** | 6-layer, 384-dim | ~22.7M | Full (all layers) | No |
| **DistilBERT** | 6-layer, 768-dim | ~66M | Full (all layers) | No |
| **BERT** | 12-layer, 768-dim | ~110M | Full (all layers) | Yes |

### Training Configuration

All three models were trained using:
- **Full fine-tuning** (all layers trainable — no frozen layers)
- **Two-group AdamW** (weight decay excluded from bias + LayerNorm)
- **Inverse-frequency class-weighted CrossEntropyLoss** (handling 13:1 class imbalance)
- **Mixed precision (FP16)** on RTX 3060 Laptop with Ampere Tensor Cores
- **Per-model batch configs** (effective batch size = 128 for all)
- **Gradient clipping** at `max_grad_norm = 1.0`
- **Linear warmup** (6% of total steps)
- **Early stopping** (patience=3 on validation Macro-F1)
- **Gradient checkpointing** for BERT-base
- **Post-training temperature scaling** for calibrated confidence probabilities
- **Seed = 42** for reproducibility

### Confidence Thresholds (Locked)

| Tier | Range | Action |
|------|-------|--------|
| LOW | < 50% | Passive monitoring + light rate limiting |
| MEDIUM | 50–80% | Aggressive throttling + CAPTCHA challenge |
| HIGH | > 80% | Immediate blocking + firewall rule update |

These thresholds operate on **calibrated probabilities** (post-temperature-scaling), not raw softmax.

### Versioned Artifacts

All training artifacts saved to `model_registry/staging/{model_name}_{dataset_version}_{timestamp}/`:
- `checkpoint/` — Model + tokenizer (HuggingFace format)
- `training_log.json` — Per-epoch metrics + VRAM peak
- `eval_report.json` — Per-class P/R/F1, confusion matrix, FPR
- `calibration/` — Temperature T, ECE before/after, reliability diagram
- `config_used.yaml` — Exact training config
- `git_hash.txt` — Repository state at training time

In [ ]:
# --- Final GPU memory cleanup ---
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e6:.1f} MB")
    print(f"GPU memory cached   : {torch.cuda.memory_reserved()/1e6:.1f} MB")

print("\nNotebook execution complete.")